In [1]:
import time
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [2]:
# ==================================================
# 1. PREPROCESSING DATA
# ==================================================

df = pd.read_csv('../data/komdigi_hoaks.csv')

df['teks'] = (
    df['title'].fillna('') + ' ' +
    df['excerpt'].fillna('')
)

df = df[df['teks'].str.strip() != ''].reset_index(drop=True)

df = df.dropna(subset=['topics']).copy()
df['topics'] = df['topics'].astype(str).str.strip()
df = df[df['topics'] != ''].copy()

# Menghapus kelas yang memiliki kurang dari 2 data
counts = df['topics'].value_counts()
df = df[df['topics'].isin(counts[counts >= 2].index)]

print(f"Total data bersih: {len(df)} baris")

X = df['teks']
y = df['topics']

# ==================================================
# 2. PEMBAGIAN DATA
# ==================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Data training: {len(X_train)}")
print(f"Data testing : {len(X_test)}")

# ==================================================
# 3. EKSTRAKSI FITUR
# ==================================================

vectorizer = CountVectorizer(
    lowercase=True,
    binary=True,
    ngram_range=(1, 2),
    max_features=5000
)

X_train_vector = vectorizer.fit_transform(X_train)
X_test_vector = vectorizer.transform(X_test)

# ==================================================
# 4. PELATIHAN MODEL
# ==================================================

model = BernoulliNB(alpha=1.0)

start = time.perf_counter()
model.fit(X_train_vector, y_train)
training_time = time.perf_counter() - start

# ==================================================
# 5. PREDIKSI
# ==================================================

start = time.perf_counter()
y_pred = model.predict(X_test_vector)
prediction_time = time.perf_counter() - start

# ==================================================
# 6. EVALUASI
# ==================================================

print("\n=== BERNOULLI NAIVE BAYES ===")

print(f"Accuracy          : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision Macro   : {precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"Recall Macro      : {recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"F1-Score Macro    : {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"F1-Score Weighted : {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Waktu Training    : {training_time:.4f} detik")
print(f"Waktu Prediksi    : {prediction_time:.4f} detik")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Total data bersih: 4176 baris
Data training: 3340
Data testing : 836

=== BERNOULLI NAIVE BAYES ===
Accuracy          : 0.8385
Precision Macro   : 0.2094
Recall Macro      : 0.1914
F1-Score Macro    : 0.1922
F1-Score Weighted : 0.7893
Waktu Training    : 0.0103 detik
Waktu Prediksi    : 0.0011 detik

Classification Report:
                    precision    recall  f1-score   support

             Hoaks       0.86      0.98      0.91       708
         Kesehatan       0.00      0.00      0.00        11
Masyarakat Digital       0.00      0.00      0.00        20
    Pejabat Publik       0.00      0.00      0.00        27
          Penipuan       0.40      0.17      0.24        58
 Program/Kebijakan       0.00      0.00      0.00        12

          accuracy                           0.84       836
         macro avg       0.21      0.19      0.19       836
      weighted avg       0.75      0.84      0.79       836

